In [1]:
import numpy as np
import time
import cProfile
import pstats



# Exercises

For the exams work on at least two of the these three exercises

## Q3: How do professionals do it?

Pick a big git repo, say [scipy](https://github.com/scipy/scipy) or [numpy](https://github.com/numpy/numpy),  and have a look at their development workflow, including their testing strategies.

I looked at the NumPy repository, mainly at:
- `doc/TESTS.rst`
- `numpy/_core/tests/`
- `pytest.ini`
- `.github/workflows/`

#### Testing
- NumPy uses `pytest`.
- Tests are organized inside the individual subpackages, in `tests/` directories.
- Naming conventions are used for automatic discovery:
  - files: `test_*.py`
  - functions: `test_*`
  - classes: `Test*`
- The test suite includes unit tests and regression tests.
- When a bug is fixed, a specific test is added so that the same problem is detected if it appears again.
- The full suite can be run with:
  `spin test -m full`
- A specific part can be tested separately, for example:
  `spin test -t numpy/_core/tests`

#### Development workflow
- Changes are proposed through pull requests.
- CI is configured in `.github/workflows/`.
- Pull requests automatically trigger build and test jobs.
- Tests are run in different configurations before merge.
- Code is reviewed after the automated checks.

So the workflow is roughly:

`code change -> tests -> pull request -> CI -> review -> merge`

## Q4: It's over Anakin, I have the high ground. I hate you! 

Pick a piece of your code that your really hate.  You hate it because it's soooooo slooooow. 
- Profile it.
- Find the culprit
- Rewrite that bottleneck making a better use of numpy arrays, or jitting it with Numba.
- Enjoy your faster code

**You underestimate my power**.

In [9]:
np.random.seed(42)

N = 500_000

pt1  = np.random.uniform(20, 100, N)
eta1 = np.random.uniform(-2.4, 2.4, N)
phi1 = np.random.uniform(-np.pi, np.pi, N)

pt2  = np.random.uniform(20, 100, N)
eta2 = np.random.uniform(-2.4, 2.4, N)
phi2 = np.random.uniform(-np.pi, np.pi, N)

In [10]:
def invariant_mass_slow(pt1, eta1, phi1,
                        pt2, eta2, phi2):

    masses = []

    for i in range(len(pt1)):

        px1 = pt1[i] * np.cos(phi1[i])
        py1 = pt1[i] * np.sin(phi1[i])
        pz1 = pt1[i] * np.sinh(eta1[i])
        e1  = pt1[i] * np.cosh(eta1[i])

        px2 = pt2[i] * np.cos(phi2[i])
        py2 = pt2[i] * np.sin(phi2[i])
        pz2 = pt2[i] * np.sinh(eta2[i])
        e2  = pt2[i] * np.cosh(eta2[i])

        E  = e1 + e2
        px = px1 + px2
        py = py1 + py2
        pz = pz1 + pz2

        m2 = E**2 - px**2 - py**2 - pz**2

        masses.append(np.sqrt(max(m2, 0)))

    return np.array(masses)

In [11]:
with cProfile.Profile() as pr:

    masses_slow = invariant_mass_slow(
        pt1, eta1, phi1,
        pt2, eta2, phi2
    )

pr.dump_stats("mass_profile.prof")

In [12]:
stats = pstats.Stats("mass_profile.prof")

stats.sort_stats("cumtime").print_stats(10)

Sat Oct 17 02:39:34 2026    mass_profile.prof

         1000816 function calls (1000798 primitive calls) in 8.085 seconds

   Ordered by: cumulative time
   List reduced from 177 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        6    0.000    0.000    8.071    1.345 /cvmfs/sft.cern.ch/lcg/releases/Python/3.13.11-41c2c/x86_64-el9-gcc13-opt/lib/python3.13/asyncio/base_events.py:1970(_run_once)
        1    0.000    0.000    8.059    8.059 /cvmfs/sft.cern.ch/lcg/releases/Python/3.13.11-41c2c/x86_64-el9-gcc13-opt/lib/python3.13/threading.py:641(wait)
        7    6.817    0.974    7.037    1.005 {built-in method time.sleep}
        1    0.420    0.420    0.458    0.458 /tmp/ipykernel_5325/2339235608.py:1(invariant_mass_slow)
   500000    0.131    0.000    0.131    0.000 {built-in method builtins.max}
   500006    0.120    0.000    0.120    0.000 {method 'append' of 'list' objects}
        1    0.025    0.025    0.025    0.025 {bu

In [13]:
def invariant_mass_fast(pt1, eta1, phi1,
                        pt2, eta2, phi2):

    px1 = pt1 * np.cos(phi1)
    py1 = pt1 * np.sin(phi1)
    pz1 = pt1 * np.sinh(eta1)
    e1  = pt1 * np.cosh(eta1)

    px2 = pt2 * np.cos(phi2)
    py2 = pt2 * np.sin(phi2)
    pz2 = pt2 * np.sinh(eta2)
    e2  = pt2 * np.cosh(eta2)

    E  = e1 + e2
    px = px1 + px2
    py = py1 + py2
    pz = pz1 + pz2

    m2 = E**2 - px**2 - py**2 - pz**2

    return np.sqrt(np.maximum(m2, 0))

In [14]:
masses_fast = invariant_mass_fast(
    pt1, eta1, phi1,
    pt2, eta2, phi2
)

print(
    np.allclose(
        masses_slow,
        masses_fast
    )
)

True


In [16]:
start = time.perf_counter()

masses_slow = invariant_mass_slow(
    pt1, eta1, phi1,
    pt2, eta2, phi2
)

t_slow = time.perf_counter() - start


start = time.perf_counter()

masses_fast = invariant_mass_fast(
    pt1, eta1, phi1,
    pt2, eta2, phi2
)

t_fast = time.perf_counter() - start


print("Slow version:", t_slow, "s")
print("Fast version:", t_fast, "s")
print("Speed-up:", t_slow / t_fast)

Slow version: 6.786844376008958 s
Fast version: 0.058243501000106335 s
Speed-up: 116.52535063090673
